[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/phase_change/phase_change.ipynb)

# Boiling at Fixed Pressure

This example computes how a van der Waals fluid boils: heat taken in at fixed pressure warms the liquid, then turns it into vapour at a fixed temperature, then warms the vapour. Below the critical point, liquid and vapour coexist at the two volumes where their pressures and their chemical potentials are equal, and the heat that boiling takes is the jump in enthalpy between them, the latent heat.

A state of coexistence is modelled as the two phases' volumes, an antivector of the plane spanned by the liquid's volume `mv.l.dual()` and the vapour's `mv.g.dual()`, each read back by pairing, `mv.l & state`. How far a state is from coexisting is a function from states to states, and its derivative at a state is a map from a correction of the state to the change it makes; Newton's method solves that map for the correction that cancels the mismatch. The solve runs over a batch of temperatures as over one, and so draws the whole dome of coexisting volumes at once.

In the function notation of thermodynamics, where each quantity is written as a function of the volume along an isotherm, coexistence is Maxwell's construction: $p(V_l) = p(V_g)$ and $\mu(V_l) = \mu(V_g)$ with $\mu = F + pV$, which cuts the isotherm's loop at the saturation pressure into two equal areas, $\int_{V_l}^{V_g} p\,dV = p_\text{sat}(V_g - V_l)$. The heat taken in at fixed pressure is $dQ = dH$, and the latent heat is $L = H_g - H_l = T(S_g - S_l)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import jax.numpy as jnp
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.phase_change import render
from examples.thermodynamics.phase_change.scenarios import stage_frames

ga = Algebra("l0g0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
State = ga.gatype.antivector()                             # Antivector-

# The two readings a state of coexistence is described by.
liquid_volume = mv.l & State                               # [] Scalar+ <- State-
vapour_volume = mv.g & State                               # [] Scalar+ <- State-

# The fluid: its heat capacity at fixed volume, the strength of its molecules' attraction and the volume they exclude,
# for a fixed amount of the fluid. Temperature and volume are in their critical values, energy in the gas constant
# times the critical temperature and pressure in that over the critical volume, so the critical pressure is 3/8.
capacity, attraction, exclusion = 1.5, 9 / 8, 1 / 3

## Liquid and vapour in coexistence

The fluid's pressure falls as its volume grows, except along a loop below the critical temperature where a uniform fluid would be unstable and splits into liquid and vapour instead. The two phases coexist at the volumes where they have the same pressure and the same chemical potential, `helmholtz_energy + pressure * volume`, two conditions that say together that the Helmholtz energy's tangent lines at the two volumes are one line, the common tangent.

This cell takes about five seconds, most of it compiling the Newton solves and drawing the charts.

In [ ]:
n_newton = 8
boiling_temperature = 0.85

def pressure(volume: Scalar, temperature: np.ndarray) -> Scalar:
    """The van der Waals equation of state."""
    return temperature / (volume - exclusion) - attraction / volume.squared()

def helmholtz_energy(volume: Scalar, temperature: np.ndarray) -> Scalar:
    """The Helmholtz energy, whose slope along the volume is minus the pressure."""
    return (-capacity * temperature * jnp.log(temperature) - temperature * (volume - exclusion).log()
            - attraction / volume)

def chemical_potential(volume: Scalar, temperature: np.ndarray) -> Scalar:
    """Where the Helmholtz energy's tangent at a volume crosses zero volume."""
    return helmholtz_energy(volume, temperature) + pressure(volume, temperature) * volume

def mismatch(state: State, temperature: np.ndarray) -> State:
    """The difference in pressure between the state's two volumes, along the liquid's, and in chemical potential,
    along the vapour's."""
    liquid, vapour = liquid_volume(state), vapour_volume(state)
    pressure_gap = pressure(liquid, temperature) - pressure(vapour, temperature)
    chemical_gap = chemical_potential(liquid, temperature) - chemical_potential(vapour, temperature)
    return mv.l.dual() * pressure_gap + mv.g.dual() * chemical_gap

# Compiled, since the charts and the heating solve it again and again.
@jax.jit
def coexistence(temperature: np.ndarray) -> State:
    """The coexisting volumes at each temperature `[...]`, by Newton's method."""
    # Below the critical point the two volumes part as the square root of the distance from it, the liquid's
    # slowly, the vapour's fast; this starts each on its own side.
    below = jnp.sqrt(1 - temperature)
    state = mv.l.dual() / (1 + 2 * below) + mv.g.dual() / (1 - below) ** 3     # [...] State-
    gap = lambda state: mismatch(state, temperature)
    for _ in range(n_newton):
        state = state - derivative(gap)(state).solve(gap(state))               # [...] State-
    return state

# The charts' second reading: the enthalpy, the heat taken in at fixed pressure being its change, and the equation of
# state solved for the temperature, which traces an isobar.
def enthalpy(volume: Scalar, temperature: np.ndarray) -> Scalar:
    return capacity * temperature - attraction / volume + pressure(volume, temperature) * volume

def isobar_temperature(volume: Scalar, isobar_pressure: Scalar) -> Scalar:
    return (isobar_pressure + attraction / volume.squared()) * (volume - exclusion)

render.draw_charts(pressure, enthalpy, coexistence, isobar_temperature, boiling_temperature)

The pressure–volume chart on the left holds isotherms, red, under the dome of coexisting volumes, black: the liquid's volumes up its left side to the critical point and the vapour's down its right. The bold isotherm at the boiling temperature crosses its loop at the saturation pressure, which cuts off two equal areas, blue below and orange above. The temperature–enthalpy chart in the middle holds the dome again and the isobars, purple, each rising through the liquid, flat across the dome while the fluid boils and rising through the vapour; the grey-blue lines across the dome are those of equal vapour fraction. The phase diagram on the right holds the saturation pressure against the temperature, the dome's flat stretches seen edge on, ending at the critical point, and lines of equal volume, green.

## Heating at fixed pressure

A weight on the piston holds the pressure at the saturation pressure of the boiling temperature, and heat comes in at a steady rate. The heat taken in at fixed pressure is the enthalpy's change, so the fluid walks along its isobar in equal steps of enthalpy. The liquid warms to the boiling temperature, where it turns into vapour without warming, its volume and enthalpy shared out between the two phases by the vapour's fraction; then the vapour warms. Where the isobar reaches a temperature, Newton's method finds the volume from the equation of state solved for the temperature.

This cell takes about five seconds, most of it drawing the animation; the Newton solves are compiled.

In [ ]:
coldest, hottest = 0.7, 0.98
n_frames, frame_duration_ms = 60, 70

@jax.jit
def volume_at(temperature: np.ndarray, isobar_pressure: Scalar, guess: Scalar) -> Scalar:
    """The volume where the isobar at a pressure reaches each temperature, from a guess on the liquid's or the vapour's
    side."""
    gap = lambda volume: isobar_temperature(volume, isobar_pressure) - temperature
    volume = guess
    for _ in range(n_newton):
        volume = volume - derivative(gap)(volume).solve(gap(volume))         # [...] Scalar+
    return volume

def mixture(coexisting: State, vapour_fraction: float, temperature: float) -> tuple[Scalar, Scalar, Scalar]:
    """The volume and the enthalpy of a mixture with a fraction of its amount as vapour, each shared out between the
    phases by that fraction, and the volume its liquid takes up."""
    liquid, vapour = liquid_volume(coexisting), vapour_volume(coexisting)
    volume = (1 - vapour_fraction) * liquid + vapour_fraction * vapour
    shared_enthalpy = ((1 - vapour_fraction) * enthalpy(liquid, temperature)
                       + vapour_fraction * enthalpy(vapour, temperature))
    return volume, shared_enthalpy, (1 - vapour_fraction) * liquid

coexisting = coexistence(np.array(boiling_temperature))                      # [] State-
liquid, vapour = liquid_volume(coexisting), vapour_volume(coexisting)        # [] Scalar+, Scalar+
saturation_pressure = pressure(liquid, boiling_temperature)                  # [] Scalar+

def heating():
    """The fluid's volume, temperature, enthalpy and the volume its liquid takes up, in equal steps of heat."""
    # The enthalpies where the stages start and end: each stage takes steps in proportion to its share of the heat.
    ends = stack([enthalpy(volume_at(coldest, saturation_pressure, liquid), coldest),
                  enthalpy(liquid, boiling_temperature), enthalpy(vapour, boiling_temperature),
                  enthalpy(volume_at(hottest, saturation_pressure, vapour), hottest)])   # [stages + 1] Scalar+
    counts = stage_frames(ends, n_frames)
    for temperature in np.linspace(coldest, boiling_temperature, counts[0], endpoint=False):
        volume = volume_at(temperature, saturation_pressure, liquid)
        yield volume, mv.scalar([temperature]), enthalpy(volume, temperature), volume
    for vapour_fraction in np.linspace(0, 1, counts[1], endpoint=False):
        volume, shared_enthalpy, liquid_share = mixture(coexisting, vapour_fraction, boiling_temperature)
        yield volume, mv.scalar([boiling_temperature]), shared_enthalpy, liquid_share
    for temperature in np.linspace(boiling_temperature, hottest, counts[2]):
        volume = volume_at(temperature, saturation_pressure, vapour)
        yield volume, mv.scalar([temperature]), enthalpy(volume, temperature), volume * 0

show_animation(render.heating(pressure, enthalpy, coexistence, isobar_temperature, boiling_temperature, heating()),
               "phase_change", duration_ms=frame_duration_ms)

The liquid at the bottom of the cylinder warms with hardly a change of volume; at the boiling temperature the vapour above it grows and lifts the loaded piston while the temperature holds, the state running along the flat stretch on the left and the flat stretch of the bold isobar in the middle, and sitting on the saturation curve in the phase diagram; once the liquid is gone the vapour warms and expands. The flat stretch in the middle is as long as the latent heat, and the steady heating spends most of its time there.

In [ ]:
latent_heat = enthalpy(vapour, boiling_temperature) - enthalpy(liquid, boiling_temperature)   # [] Scalar+

In [ ]:
print("coexisting volumes:", liquid.to_array(), vapour.to_array())
print("saturation pressure:", saturation_pressure.to_array())
print("latent heat:", latent_heat.to_array())

In [ ]:
# checks
np.testing.assert_allclose(mismatch(coexisting, boiling_temperature).kernel, 0, atol=1e-10)
# The latent heat is the temperature times the jump in entropy, the logarithm of the free volumes' ratio.
np.testing.assert_allclose(latent_heat.kernel, (boiling_temperature * ((vapour - exclusion) / (liquid - exclusion)).log()).kernel, atol=1e-10)
# The isobar's volume at a temperature has the isobar's pressure.
isobar_volume = volume_at(coldest, saturation_pressure, liquid)
np.testing.assert_allclose((pressure(isobar_volume, coldest) - saturation_pressure).kernel, 0, atol=1e-10)
# Clapeyron: the saturation pressure rises with the temperature as the latent heat over the temperature and the jump in volume.
step = 1e-4
rise = (pressure(liquid_volume(coexistence(np.array(boiling_temperature + step))), boiling_temperature + step)
        - pressure(liquid_volume(coexistence(np.array(boiling_temperature - step))), boiling_temperature - step)) / (2 * step)
np.testing.assert_allclose(rise.kernel, (latent_heat / (boiling_temperature * (vapour - liquid))).kernel, rtol=1e-6)